# NB05 — Cython Basics for Computational Biology

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

Numba works by JIT-compiling at runtime. Cython works differently: it is a **superset of Python** that you compile to C ahead of time, then import as a Python extension module. Cython is how NumPy, scikit-learn, Biopython, and many other core scientific Python packages accelerate their inner loops.

You need to understand Cython to:
- Read the source of packages you depend on (e.g., `scipy/_optimize/_lbfgsb.pyx`)
- Write a fast inner loop for a tool where Numba is not applicable (e.g., complex Python object manipulation)
- Understand the performance model of extensions you use

---

## 2. Intuition

Pure Python is slow because every operation involves:
1. A Python object with a reference count
2. A dynamic type check at runtime
3. A Python interpreter dispatch

Cython lets you **declare types**: `cdef int i`, `cdef double gc`. Once a variable has a declared C type, Cython compiles that computation directly to C — no Python objects, no type checks, no interpreter overhead. The result is indistinguishable from hand-written C.

**Typed memoryviews** (`double[:]`, `int8_t[:, :]`) give direct pointer access to NumPy array data — no bounds checking overhead, no Python object creation per element.

---

## 3. Biological background

Biopython's pairwise alignment code uses Cython internally. The core alignment loop (filling a DP matrix) is in a `.pyx` file that looks almost like Python but compiles to C.

K-mer counting is another canonical Cython use case: the inner loop (slide a window over bytes, hash it) is a perfect candidate for `cdef` typed loops with a C-level hash table.

Understanding the Cython source in packages like these is a practical bioinformatics skill — it tells you *why* a function is fast and what its edge-case behavior is.

---

## 4. Mathematical explanation

The speedup from Cython comes from removing Python overhead at the variable level:

| Operation | Pure Python | Cython typed |
|-----------|------------|---------------|
| `i += 1` | Incref/decref Python int object | C `i++` (single instruction) |
| `a[i]`   | `PyObject_GetItem` call | C array dereference (single instruction) |
| `if x > y` | Python comparison → boolean object | C `if (x > y)` |

The net effect: a Cython typed loop runs at C speed, typically 10–200x faster than the equivalent pure Python loop, depending on how much Python object overhead the loop contains.

---

## 5. Computational explanation

### Key Cython constructs

```cython
# cdef: C-level function (not callable from Python)
cdef double gc_content_c(int8_t[:] seq):
    cdef int i, n = len(seq), gc = 0
    for i in range(n):
        if seq[i] == 2 or seq[i] == 3:
            gc += 1
    return gc / n

# cpdef: C-level fast path + Python wrapper (callable from both)
cpdef double gc_content(int8_t[:] seq):
    return gc_content_c(seq)

# nogil: release the GIL — allows true thread parallelism for C code
with nogil:
    result = gc_content_c(seq)
```

### Typed memoryview syntax
```cython
def fn(double[:, :] mat):   # 2D double array, any memory layout
    cdef int i, j
    cdef double total = 0.0
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            total += mat[i, j]
    return total
```

---

## 6. Python implementation

This notebook shows Cython source code as strings and explains each line. Compilation requires a C compiler and the `cython` package. The Python reference implementations are fully executable.

In [ ]:
import numpy as np
import timeit
import matplotlib.pyplot as plt

np.random.seed(42)
print("This notebook illustrates Cython patterns without requiring compilation.")
print("To actually compile Cython, you need: pip install cython, and a C compiler.")

In [ ]:
# --- Cython source: hamming_distance ---
# Shown as a string; this would live in a .pyx file

hamming_pyx = '''
# hamming.pyx — Cython source for hamming_distance
# Compile with: cython hamming.pyx && gcc -shared -fPIC -o hamming.so hamming.c $(python3-config --includes)

import numpy as np
cimport numpy as np                          # Cython-level NumPy declarations
from numpy cimport int8_t, int32_t           # C-level integer types

# cpdef: callable from Python AND from Cython at C speed
cpdef int32_t hamming_distance(int8_t[:] a, int8_t[:] b):
    """
    Count positions where a and b differ.
    int8_t[:] is a typed memoryview: direct pointer to NumPy array data.
    """
    cdef int i                               # C-level loop variable
    cdef int32_t diff = 0                    # C-level accumulator
    cdef int n = a.shape[0]                  # C-level length
    
    if a.shape[0] != b.shape[0]:             # Python-level check
        raise ValueError("Sequences must have equal length")
    
    for i in range(n):                       # Compiled to: for (i=0; i<n; i++)
        if a[i] != b[i]:                     # C array dereference, no Python object
            diff += 1
    
    return diff
'''

print("Cython source for hamming_distance:")
print(hamming_pyx)
print()
print("Line-by-line explanation:")
print("  cimport numpy as np  — imports Cython-level NumPy declarations (not Python-level)")
print("  int8_t[:] a          — typed memoryview: no Python list, direct pointer to array data")
print("  cdef int i           — C integer, not Python int object — allocated on the C stack")
print("  for i in range(n)    — Cython compiles this to a C for loop when i is typed")
print("  a[i]                 — C array dereference when a has a typed memoryview type")

In [ ]:
# --- Cython source: kmer_count ---

kmer_pyx = '''
# kmer.pyx — Cython source for k-mer counting

from numpy cimport int8_t
import numpy as np

def kmer_count_cython(int8_t[:] seq, int k):
    """
    Count all k-mers in an integer-encoded sequence.
    Returns a Python dict {bytes: int}.
    
    Note: the dict is a Python object — the speedup comes from the
    inner loop (byte access, tuple creation) being faster, not from
    a pure-C hash table. For a fully compiled version, use a C++ map.
    """
    cdef int i, j
    cdef int n = seq.shape[0]
    cdef int8_t[:] window
    
    counts = {}                              # Python dict — still involves Python overhead
    
    for i in range(n - k + 1):              # C loop over positions
        # Typed memoryview slice — no copy, just a view
        window = seq[i:i+k]                  # C-level slice
        key = bytes(window)                  # Convert to bytes for hashing — Python object
        if key in counts:
            counts[key] += 1
        else:
            counts[key] = 1
    
    return counts
'''

print("Cython source for kmer_count:")
print(kmer_pyx)
print("Key insight: even though the dict is a Python object, the loop overhead")
print("is eliminated. Each iteration does C-speed array access + one Python dict lookup.")

In [ ]:
# --- Python reference implementations (fully executable) ---

def hamming_python_pure(a, b):
    """Pure Python: each iteration involves Python int objects."""
    if len(a) != len(b):
        raise ValueError("Sequences must have equal length")
    return sum(1 for x, y in zip(a, b) if x != y)

def hamming_numpy(a, b):
    """NumPy vectorized: no Python loop."""
    return int(np.sum(a != b))

def kmer_count_python(seq, k):
    """Pure Python k-mer counter."""
    from collections import Counter
    return Counter(tuple(seq[i:i+k]) for i in range(len(seq) - k + 1))

# Test
a = np.array([0, 1, 2, 3, 0, 1, 2], dtype=np.int8)
b = np.array([0, 1, 3, 3, 1, 1, 2], dtype=np.int8)

assert hamming_python_pure(a, b) == hamming_numpy(a, b)
print(f"Hamming distance: {hamming_numpy(a, b)}")
print("Correctness: PASSED")

seq = np.random.randint(0, 4, 100, dtype=np.int8)
kmers = kmer_count_python(seq, k=3)
print(f"K-mer count (k=3, L=100): {len(kmers)} distinct k-mers")

In [ ]:
# --- Speedup progression: simulated benchmark data ---
# Based on typical measured values in the Cython documentation
# and published benchmarks for similar loops.
# Real values vary by CPU and compiler flags.

# Actual benchmark: Python pure loop vs NumPy (we can run this)
seq_bench = np.random.randint(0, 4, 10_000, dtype=np.int8)

t_pure = timeit.timeit(
    lambda: hamming_python_pure(seq_bench, seq_bench),
    number=100
) / 100

t_numpy = timeit.timeit(
    lambda: hamming_numpy(seq_bench, seq_bench),
    number=1000
) / 1000

# Estimated Cython and C values (from literature/typical measurements)
# Cython typed = ~same as C for simple integer loops
# Cython untyped = ~2-3x faster than Python (interpreter calls removed)
t_cython_untyped_estimated = t_pure / 3.0  # approximate
t_cython_typed_estimated = t_numpy * 0.8    # similar to NumPy, no Python overhead

print(f"L=10,000 Hamming distance:")
print(f"  Pure Python:         {t_pure*1000:.3f} ms")
print(f"  NumPy vectorized:    {t_numpy*1000:.3f} ms  (speedup: {t_pure/t_numpy:.0f}x)")
print(f"  Cython untyped*:     {t_cython_untyped_estimated*1000:.3f} ms  (est. speedup: {t_pure/t_cython_untyped_estimated:.0f}x)")
print(f"  Cython typed*:       {t_cython_typed_estimated*1000:.3f} ms  (est. speedup: {t_pure/t_cython_typed_estimated:.0f}x)")
print("  * estimated from published benchmarks; actual values vary")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Cython optimization layers
ax = axes[0]
ax.axis('off')
layers = [
    ("Pure Python",          "Loop over Python objects\nDynamic type dispatch\nGIL held throughout",     '#e74c3c', 1.0),
    ("Cython (untyped)",     "Python objects but\nno interpreter dispatch\n~2-5x faster",               '#e67e22', 3.0),
    ("Cython (typed cpdef)", "C types for variables\nDirect array access\n~10-100x faster",            '#f1c40f', 30.0),
    ("Cython + nogil",       "GIL released\nTrue thread parallelism\nfor typed code possible",          '#27ae60', 50.0),
    ("Pure C",               "Hand-written C extension\nNo Python overhead at all",                     '#2980b9', 100.0),
]
for idx, (name, desc, color, speedup) in enumerate(layers):
    y = idx * 0.18
    ax.barh(y, speedup, height=0.14, color=color, alpha=0.85, edgecolor='black')
    ax.text(speedup + 1, y, f'{speedup:.0f}x  {name}\n{desc}',
            va='center', fontsize=7.5)
ax.set_xlim(0, 140)
ax.set_ylim(-0.1, len(layers) * 0.18)
ax.set_xlabel('Relative speedup vs pure Python', fontsize=10)
ax.set_title('Cython Optimization Layers\n(typical speedup, simple integer loop)', fontweight='bold')
ax.set_yticks([])

# Panel 2: Typed vs untyped speedup (bar chart)
ax = axes[1]
categories = ['Pure Python', 'NumPy', 'Cython\n(untyped, est.)', 'Cython\n(typed, est.)']
times_ms = [
    t_pure * 1000,
    t_numpy * 1000,
    t_cython_untyped_estimated * 1000,
    t_cython_typed_estimated * 1000
]
colors = ['#e74c3c', '#2980b9', '#e67e22', '#27ae60']
bars = ax.bar(categories, times_ms, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, times_ms):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            f'{v:.3f}ms', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Wall time (ms)', fontsize=11)
ax.set_title(f'Hamming Distance (L=10,000)\nExecution Time by Implementation', fontweight='bold')
ax.set_yscale('log')
ax.grid(True, axis='y', alpha=0.3)
ax.text(2.5, max(times_ms) * 0.5, '* estimated', fontsize=9, ha='center', color='gray', style='italic')

plt.tight_layout()
plt.savefig('../datasets/nb05_cython.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Read a real .pyx file:** Find `biopython/Bio/pairwise2.py` (or the Cython version if available). Identify: (a) where `cdef` appears, (b) what typed memoryviews are used, (c) where `nogil` appears if at all.

2. **Cython vs Numba decision:** For each task below, decide whether you would use Cython or Numba, and explain why:
   - Fill a DP matrix for Viterbi decoding of an HMM
   - Count k-mers in a streaming FASTA reader (using a generator)
   - Apply a custom scoring function to each row of a large NumPy matrix

3. **GC content in Cython style:** Write the `.pyx` source (as a Python string) for a function `gc_content_cython(int8_t[:] seq) -> double`. Annotate every line with a comment explaining what Cython compiles it to.

4. **The `nogil` constraint:** What types of Python operations are NOT allowed inside a `nogil` block? List three operations you might accidentally try inside `nogil` that would cause a compile error.

## 12. Reflection

*In your own words: When would you choose Cython over Numba? What does `cimport` do that `import` cannot? If you are writing a new bioinformatics tool today, what is your default strategy for speeding up Python loops?*

---

## Papers referenced

- Behnel, S. et al. (2011). "Cython: The Best of Both Worlds." *Computing in Science & Engineering* 13(2):31–39.

## References

- Cython docs: https://cython.readthedocs.io/en/latest/
- Typed memoryviews: https://cython.readthedocs.io/en/latest/src/userguide/memoryviews.html
- NumPy Cython tutorial: https://cython.readthedocs.io/en/latest/src/tutorial/numpy.html

## Future work / open questions

- Cython 3.0 introduced pure-Python mode: you can write Cython type annotations as Python type hints, compile optionally, and still run as plain Python without Cython. How does this interact with mypy?
- How does Cython's `nogil` relate to the upcoming Python 3.13 free-threaded build? Could you write a Cython extension that uses both?